In [2]:
#Environment setup and data loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, RocCurveDisplay)

#loading phase 1 artifacts
X_train=pd.read_csv("D:/DIYA/PROJECTS/customer-churn-xai/data/X_train.csv")
X_test=pd.read_csv("D:/DIYA/PROJECTS/customer-churn-xai/data/X_test.csv")
y_train=pd.read_csv("D:/DIYA/PROJECTS/customer-churn-xai/data/y_train.csv").squeeze("columns")
y_test=pd.read_csv("D:/DIYA/PROJECTS/customer-churn-xai/data/y_test.csv").squeeze("columns")

print(f"X_train: {X_train.shape} | y_train churn rate: {y_train.mean():.4f}")
print(f"X_test: {X_test.shape} | y_test churn rate: {y_test.mean():.4f}")

X_train: (5634, 35) | y_train churn rate: 0.2654
X_test: (1409, 35) | y_test churn rate: 0.2654


In [6]:
#hyperparameter tuning

#base estimator with logloss metric
xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

#focused hyperparameter search grid
param_grid={
    "n_estimators":[100,150,200],
    "max_depth":[3,4,5],
    "learning_rate":[0.03,0.05,0.1],
    "subsample":[0.8,1.0],
    "colsample_bytree":[0.8,1.0],
    "scale_pos_weight":[1,3,5]     #imbalance penalty calculated previously
}

#5-fold stratified cross validation optimizing for ROC-AUC
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search=GridSearchCV(estimator=xgb_base, param_grid=param_grid, scoring="roc_auc", cv=cv, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

best_xgb=grid_search.best_estimator_
print(f"\nBest Cross-Validation ROC-AUC: {grid_search.best_score_:.4f}")
print("Optimal Hyperparammeters:")
for param, val in grid_search.best_params_.items():
    print(f"{param}: {val}")

Fitting 5 folds for each of 324 candidates, totalling 1620 fits

Best Cross-Validation ROC-AUC: 0.8500
Optimal Hyperparammeters:
colsample_bytree: 0.8
learning_rate: 0.03
max_depth: 3
n_estimators: 200
scale_pos_weight: 1
subsample: 0.8


In [9]:
#test set evaluation- default 0.5 decision threshold

#generate probabilities and default predcitions
y_probs=best_xgb.predict_proba(X_test)[:,1]
y_pred_default=(y_probs>=0.50).astype(int)

auc_score=roc_auc_score(y_test,y_probs)
cm_default= confusion_matrix(y_test,y_pred_default)

print(f"===XGBoost Benchmark (Threshold=0.50)===")
print(f"ROC-AUC Score: {auc_score:.4f}\n")
print("Confusion Matrix:")
print(pd.DataFrame(cm_default, index=["Actual Retained(0)", "Actual Churned(1)"],
                   columns=["Pred Retained(0)", "Pred Churned(1)"]))
print(f"Classification Report:\n {classification_report(y_test, y_pred_default, digits=4)}")

===XGBoost Benchmark (Threshold=0.50)===
ROC-AUC Score: 0.8475

Confusion Matrix:
                    Pred Retained(0)  Pred Churned(1)
Actual Retained(0)               935              100
Actual Churned(1)                185              189
Classification Report:
               precision    recall  f1-score   support

           0     0.8348    0.9034    0.8677      1035
           1     0.6540    0.5053    0.5701       374

    accuracy                         0.7977      1409
   macro avg     0.7444    0.7044    0.7189      1409
weighted avg     0.7868    0.7977    0.7888      1409



In [15]:
#threshold optimization for retention call

#sweep classification thresholds between 0.20 and 0.60
thresholds=np.arange(0.20,0.65,0.05)
threshold_metrics=[]

for t in thresholds:
    preds=(y_probs>=t).astype(int)
    cm=confusion_matrix(y_test,preds)
    recall=cm[1,1]/(cm[1,0]+cm[1,1])  #TP/(FN+TP)
    precision=cm[1,1]/(cm[0,1]+cm[1,1]) if (cm[0,1]+cm[1,1]) > 0 else 0  #TP/(FP+TP)
    f1=2*recall*precision/(recall+precision) if (recall+precision) > 0 else 0
    threshold_metrics.append(
        {"threshold":round(t,2), 
         "recall (churn)":round(recall, 4),
         "precision (churn)":round(precision,4), 
         "f1-score":round(f1,4),
         "false negatives":cm[1,0],
         "false positives":cm[0,1]}
    )

threshold_df=pd.DataFrame(threshold_metrics)
print(threshold_df.to_string(index=False))

#selecting optimal threshold targeting ~75% minority recall (~0.35)
optimal_threshold=0.35
y_pred_optimal=(y_probs>=optimal_threshold).astype(int)
cm_opt=confusion_matrix(y_test,y_pred_optimal)

print(f"\n===Tuned XGBoost Performance (Threshold={optimal_threshold})===")
print(pd.DataFrame(cm_opt, index=["Actual Retained(0)", "Actual Churned(1)"],columns=["Pred Retained(0)", "Pred Churned(1)"]))
print(f"\nClassification Report:\n {classification_report(y_test, y_pred_optimal, digits=4)}")

 threshold  recall (churn)  precision (churn)  f1-score  false negatives  false positives
      0.20          0.8583             0.4700    0.6074               53              362
      0.25          0.8262             0.4984    0.6217               65              311
      0.30          0.7834             0.5149    0.6214               81              276
      0.35          0.7219             0.5637    0.6331              104              209
      0.40          0.6604             0.5952    0.6261              127              168
      0.45          0.5989             0.6382    0.6179              150              127
      0.50          0.5053             0.6540    0.5701              185              100
      0.55          0.4358             0.7056    0.5388              211               68
      0.60          0.3342             0.7310    0.4587              249               46

===Tuned XGBoost Performance (Threshold=0.35)===
                    Pred Retained(0)  Pred Churned

In [19]:
#export model and metadata for SHAP and Streamlit
import os
os.makedirs("../models", exist_ok=True)

#save trained xgboost estimator
joblib.dump(best_xgb,"../models/churn_model.pkl")
#save  threshold and feature names metadata
metadata= {"optimal_thresold":optimal_threshold, "feature_names":list(X_train.columns), "scale_pos_weight":2.77}
joblib.dump(metadata, "../models/model_metadata.pkl")

print("Artifacts successfully saved to models/churn_model.pkl and models/model_metadata.pkl")

Artifacts successfully saved to models/churn_model.pkl and models/model_metadata.pkl
